# Comparative Analysis of Focal Loss vs. Binary Cross-Entropy in Simple Linear Models
**Course Project: Class Imbalance in Machine Learning**

### Project Overview
This notebook implements an optimization framework to evaluate **Focal Loss** within a custom **Logistic Regression** classifier. While Focal Loss was originally engineered by Lin et al. (2017) to mitigate severe background-foreground pixel imbalances in deep convolutional neural networks (RetinaNet), this study explores its efficacy and mathematical edge cases when applied to rigid, linear models across 50 tabular datasets.

### Key Analytical Intentions:
1. **Custom Implementation:** Utilize `autograd` to calculate accurate mathematical gradients for an exotic, non-standard loss profile.
2. **Automated Alpha Weighting:** Dynamically assign class weights inversely proportional to target frequencies.
3. **Robust Evaluation:** Benchmark cross-validated performance using metrics resilient to imbalance ($\text{ROC-AUC}$, $\text{PR-AUC}$, $\text{F1-Score}$, and $\text{Recall}$).

## Where the code is

The model, the losses, and the cross-validation loop live in the `mla` package. This notebook only runs that code and shows the table.

Focal-loss entry point: `mla.evaluate.run_focal_loss`.

## 2. Mathematical Definition of Custom Focal Loss
Standard Binary Cross-Entropy (BCE) scales uniformly with misclassification distance. In highly imbalanced domains, thousands of easily classified majority class examples dominate the loss sum, burying the gradient signal of rare, essential cases.

Focal Loss breaks this symmetry by introducing a modulating factor $(1 - p)^\gamma$. The objective function optimized below follows:

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Where:
* $p$ represents the raw predicted probability outputted by the model's sigmoid layer.
* $\gamma$ is the **Focusing Parameter**. When $\gamma = 0$, this loss collapses identically into Binary Cross-Entropy. As $\gamma \to 2$, the loss contribution of confident predictions ($p > 0.7$) is dynamically crushed toward zero.
* $\alpha$ balances class occurrences based on relative frequencies.

In [1]:
from mla.evaluate import run_focal_loss

results = run_focal_loss()

Found 50 datasets



Focal loss:   0%|          | 0/50 [00:00<?, ?it/s]

/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/olapacia/Desktop/PL3_G2/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


... (40 outputs left out)

Focal loss:  98%|█████████▊| 49/50 [00:25<00:00,  2.74it/s]

Focal loss: 100%|██████████| 50/50 [00:25<00:00,  2.78it/s]

Focal loss: 100%|██████████| 50/50 [00:25<00:00,  1.94it/s]


Saved /Users/olapacia/Desktop/PL3_G2/output/focal_loss_results.csv

Focal loss (gamma=2.0)
                            dataset  imbalance_ratio  alpha_pos  roc_auc_mean  pr_auc_mean  f1_mean  recall_mean
           dataset_1000_hypothyroid            11.96     0.9229        0.5299       0.9255   0.8936       0.8604
     dataset_1002_ipums_la_98-small             8.46     0.8943        0.5040       0.8918   0.8005       0.7317
     dataset_1004_synthetic_control             5.00     0.8333        0.8413       0.6483   0.4520       0.8100
dataset_1013_analcatdata_challenger            14.33     0.9348        0.4629       0.1058   0.0711       0.4000
      dataset_1014_analcatdata_dmft             4.14     0.8055        0.4907       0.2360   0.2395       0.4903
                 dataset_1016_vowel            10.00     0.9091        0.4570       0.1687   0.1321       0.4222
     dataset_1018_ipums_la_99-small            14.57     0.9358        0.4939       0.9314   0.8753       0.8213
    

## 7. Post-Mortem Analysis: Why Advanced Focal Loss Degraded the Linear Baseline
Reviewing the global performance results yields an important insight: the **Focal Loss algorithm failed to outperform standard Cross-Entropy** ($\text{ROC-AUC}$ fell from **0.706** to **0.557**; $\text{PR-AUC}$ dropped from **0.571** to **0.440**). 

This is not a failure of code implementation. It provides an exceptional case study highlighting architectural mismatches in applied machine learning:

1. **The Core Linearity Constraint:** Focal loss scales gradient updates to shift the geometric focus of training weights, but it cannot expand model capacity. Logistic Regression can only learn a completely flat hyper-plane decision boundary. If target classes are heavily mixed, non-linear, or nested, modifying the loss values cannot resolve the geometric overlap.
2. **Optimization Gradients and Convergence Rates:** The focal mathematical factor $(1-p)^\gamma$ severely compresses gradients for easily identifiable instances. For a standard first-order optimization technique running for only 500 iterations at a step size of $\eta = 0.01$, these tiny gradients stall convergence, preventing the model from stabilizing or tracking stable boundaries.
3. **The Alpha ($\alpha$) Over-Correction Penalty Loop:** Combining automatic sample scaling weights with structural focal parameters creates extreme optimization bias. The model attempts to escape intense penalty flags by aggressively predicting the rare minority class everywhere, which breaks downstream precision metrics without retrieving any meaningful boost in overall system recall.